# Real Estate Market Analysis

***Introduction***

**Source Data**: The dataset is provided by the Yandex Real Estate service and contains an archive of apartment sales listings in Saint Petersburg and the surrounding areas collected over several years.

**Goal**: The objective is to identify the key parameters that influence the market value of real estate. The findings from this Exploratory Data Analysis (EDA) will be used to build an automated system capable of detecting anomalies and fraudulent activity in the listings.

The dataset includes two types of data for each property:

- User-generated data: Information entered manually by the seller.
- Automated (Spatial) data: Information calculated automatically based on map data (e.g., distance to the city center, airport, nearest parks, and bodies of water).

**Key Business Questions**:

- *Time to Sale*: How long does it typically take to sell an apartment? At what point is a sale considered "too fast" or "abnormally slow"?
- *Price Drivers*: Which factors have the strongest impact on the total market value of an apartment?
- *Location Analysis*: What is the average price per square meter in the top 10 locations with the highest number of listings?
- *City Center vs. Suburbs*: Which factors influence prices specifically in the Saint Petersburg city center, and how do they differ from the overall market trends?

## 1. Examining data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
data = pr.read_csv('real_estate_data.csv', sep='\t')

In [ ]:
pd.set_option('display.max_columns', None)
data.head(20)

In [ ]:
data.info()

In [ ]:
data.hist(figsize=(15,20)); #plotting histograms for all numeric columns

**Initial Data Inspection: Key Findings**

1. *Data Types:* The columns `floors_total`, `balcony`, `parks_around3000`, and `ponds_around3000` need to be converted to **integer** type. The `is_apartment` column should be **boolean**, and `first_day_exposition` needs to be converted to **datetime**.
2. *Style:* The `cityCenters_nearest` column name needs to be renamed to follow **snake_case** conventions.
3. *Missing Values:* There are missing values in several columns.
- *Human Factor:* Users may have left fields blank to imply a "zero" value (e.g., leaving the number of balconies blank because they don't have one).
- *Technical Issues:* Some missing data might be due to errors in the automated map generation process.
4. *Anomalies:* There are clear outliers and anomalous values, such as properties with 0 rooms in the `rooms` column or ceiling heights of 100 meters in the `ceiling_height` column.

**Data Description:**

- `total_images`(int64) - number of photos in the listing,
- `last_price`(float64) - price at the time the listing was removed,
- `total_area`(float64) - total apartment area in square meters (m²),
- `first_day_exposition`(object) - date of publication,
- `rooms`(int64) - number of rooms,
- `ceiling_height`(float64) - ceiling height (m),
- `floors_total`(float64) - total number of floors in the building,
- `living_area`(float64) - living area in square meters (m²),
- `floor`(int64) - the floor the apartment is located on,
- `is_apartment`(object) - apartment status,
- `studio`(bool) - whether the apartment is a studio,
- `open_plan`(bool) - whether the apartment has an open floor plan,
- `kitchen_area`(float64) - kitchen area in square meters (m²),
- `balcony`(float64) - number of balconies,
- `locality_name`(object) - name of the locality/settlement,
- `airports_nearest`(float64) - distance to the nearest airport (m),
- `cityCenters_nearest`(float64) - distance to the city center (m),
- `parks_around3000`(float64) - number of parks within a 3 km radius,
- `parks_nearest`(float64) - distance to the nearest park (m),
- `ponds_around3000`(float64) - number of bodies of water within a 3 km radius,
- `ponds_nearest`(float64) - distance to the nearest body of water (m),
- `days_exposition`(float64) - number of days the ad was listed (from publication to removal).

## 2. Data preprocessing

### 2.1 Gaps removal

In [ ]:
data = data.rename(columns={'cityCenters_nearest':'city_centers_nearest'})

In [ ]:
data.isna().sum() # count of missing values for each column

In [ ]:
# balcony - filling 0s in gaps
data['balcony'] = data['balcony'].fillna(0)
data['balcony'].isna().velue_counts()

In [ ]:
#is_apartment - filling gaps with false values
data['is_apartment'] = data['is_apartment'].fillna(value=False)
data['is_apartment'].isna().value_counts()

In [ ]:
#ceiling_height - filling with median values
ceiling_height_avg = data['ceiling_height'].median()
data['ceiling_height'] = data['ceiling_height'].fillna(value=ceiling_height_avg)
data['ceiling_height'].isna().value_counts()

In [ ]:
#kitchen_area & living_area

# ratio of kitchen area to total area
data['kitchen_ratio'] = data['kitchen_area'] / data['total_area']
# ratio of living area to total area
data['living_ratio'] = data['living_area'] / data['total_area']

In [ ]:
# average area ratios typical for apartments
rates = data.pivot_table(index='rooms', values=['living_ratio', 'kitchen_ratio'], aggfunc='median').median()

#filling missing area values based on the ratios calculated in the previous step
#ensuring living + kitchen area does not exceed 90% of the total area (allowing 10% for utility spaces like hallways and bathrooms)
data.loc[(data['living_area'].isna()) | (data['kitchen_area'].isna()) |
    ((data['kitchen_area'] + data['living_area']) > data['total_area'] * 0.9), ['living_area', 'kitchen_area']] =\
    pd.DataFrame({'living_area': data['total_area'] * rates['living_ratio'], 'kitchen_area': data['total_area'] * 
                 rates['kitchen_ratio']})

In [ ]:
data['living_area'].isna().value_counts()

In [ ]:
data['litchen_area'].isna().value_counts()

In [ ]:
# locality_name - dropping 49 rows with missing values
data = data.dropna(subset=['locality_name']).reset_index(drop=True)
data['locality_name'].isna().value_counts()

In [ ]:
# floors_total - filling 86 missing values with the current floor number (ensuring a minimum of 5 floors)
data.loc[data['floors_total'].isna(), 'floors_total'] = data['floor'].where(data['floor'] > 4, 5)
data['floors_total'].isna().value_counts()

In [ ]:
data.isna().sum()

In [ ]:
data.info()

***Missing Value Analysis & Handling Strategy***

The following missing values were identified and their potential causes were analyzed:

**1. Handled / Imputed Variables:**

- `balcony`: The seller likely left this blank because the apartment has no balcony. This is a human factor; I treated these missing values as 0.
- `parks_around3000` & `ponds_around3000`: These fields are populated automatically (not by the user), suggesting a technical error in the data generation process or that these fields are optional in the system.
- `is_apartment`: In this context, "apartments" refer to non-residential commercial properties (without permanent residency rights). If the seller skipped this field (human factor), it is safe to assume the property is standard residential housing. I filled these with False.
- `ceiling_height`: Likely missing due to the human factor; sellers may have considered this detail optional or low priority.
- `kitchen_area` & `living_area`: Likely due to the human factor. Sellers often provide only the `total_area` without breaking it down, or they estimate room sizes "by eye." This sometimes leads to anomalies where the sum of the kitchen and living areas exceeds the total area.
- `floors_total`: This is user-entered data (human factor) and was likely treated as optional by some sellers.
- `locality_name`: This is a critical field; without the location, the rest of the data loses its value. These rows might result from a database extraction error. Since I lack coordinates to restore the location, I decided to drop these rows.

**2. Variables Left Unchanged:**

- `parks_nearest`, `ponds_nearest`, `airports_nearest` & `city_centers_nearest`:

*Reason*: This is spatial data calculated automatically based on maps. The missing values are likely a technical issue (e.g., the system only calculates these for specific types of settlements).

*Decision*: I decided not to fill these gaps. Imputing distances without precise coordinates would distort the data.

- `days_exposition`:

*Reason*: This field shows how long an ad was active before being closed (sold). A missing value (NaN) likely indicates that the ad is still active (not yet sold).

*Decision*: Filling this with an arbitrary value would create anomalies, so I left it as is.

### 2.2 Data Types Conversion

The data types for the following columns will be changed:

- `floors_total`, `balcony`, `parks_around3000`, and `ponds_around3000` will be converted to *int*.
- `first_day_exposition` will be converted to *datetime*.

In [ ]:
data['floors_total'] = data['floors_total'].astype('int')

In [ ]:
data['balcony'] = data['balcony'].astype('int')

In [ ]:
data['parks_around3000'] = data['parks_around3000'].astype('int')

In [ ]:
data['ponds_around3000'] = data['ponds_around3000'].astype('int')

In [ ]:
data['first_day_exposition'] = pd.to_datetime(data['first_day_exposition'], format='%Y-%m-%dT%H:%M:%S')

In [ ]:
data.head(20)

In [ ]:
data.info()

In this section, the data types were updated for specific columns based on the following logic:

- `is_apartment`: After filling missing values (NaN) with False in the previous step, the column automatically converted from object to bool.
- `parks_around3000` & `ponds_around3000`: Since it is logically impossible to have "2.5 parks" or "1.5 ponds," I converted these variables to integers.
- `floors_total` & `balcony`: Similarly, a building cannot have "10.3 floors" or "1.5 balconies." Therefore, I converted these columns to int64.
- `first_day_exposition`: Standard change of the date strings into the datetime format.

### 2.3 Duplicates handling

In [ ]:
data['locality_name'].unique()

In [ ]:
len(data['locality_name'].unique())

Upon inspecting the unique values, implicit duplicates were identified that need to be eliminated. These are primarily caused by spelling inconsistencies (e.g., using 'е' vs. 'ё') and varying naming conventions.

*Examples:*

- 'поселок' vs. 'посёлок'
- 'городской посёлок Янино-1' vs. 'городской поселок Янино-1'
- 'посёлок Мурино' vs. 'поселок Мурино' vs. 'Мурино'

In [ ]:
# applying vectorized string splitting to eliminate implicit duplicates in the locality_name column
data.loc[:, ['locality_type', 'locality_clean_name']] =\
    data['locality_name'].str.split(pat='(?=[A-Я])', n=1, expand=True).to_numpy()

In [ ]:
data.head(20)

In [ ]:
data['locality_type'].unique()

In [ ]:
# eliminated implicit duplicates in locality types
data['locality_type'] = (
    data['locality_type']
    .str.replace('ё', 'е')
    .replace(['городской поселок', 'поселок городского типа имени', 'поселок городского типа', 'поселок станции',
             'поселок при железнодорожной станции', 'коттеджный поселок'], 'поселок'))

In [ ]:
data['locality_type'].unique()

In [ ]:
data['locality_clean_name'].unique()

In [ ]:
len(data['locality_clean_name'].unique())

In [ ]:
data.info()

**Conclusion:** Separating the locality types from the locality names allowed to extract a column of "clean" names, effectively eliminating the majority of implicit duplicates. The locality types were also standardized to remove inconsistencies.

### 2.4 Handling abnormal values

- **`ceiling_height` variable**

In [ ]:
len(data['ceiling_height'].unique())

In [ ]:
data.boxplot(column='ceiling_height');

In [ ]:
data.boxplot(column='ceiling_height', showfliers=False);

In [ ]:
data['ceiling_height'].sort_values(ascending=False).unique()

In [ ]:
data['ceiling_height'].describe()

In [ ]:
# handling ceiling height outliers (values > 10)
data.loc[data['ceiling_height'] > 10, 'ceiling_height'] /= 10

In [ ]:
data['ceiling_height'].sort_values(ascending=False).unique()

In [ ]:
data[(data['ceiling_height'] > 2.85) & (data['ceiling_height'] < 4)]['ceiling_height'].hist(bins=30);

In [ ]:
# removing values outside the range
data = data.query('2.45 <= ceiling_height <= 3.6').reset_index(drop=True)

In [ ]:
data['ceiling_height'].sort_values(ascending=False).unique()

In [ ]:
data.boxplot(column='ceiling_height');

In [ ]:
len(data['ceiling_height'].unique())

In [ ]:
data.info()

**Conclusion:** The `ceiling_height` variable was analyzed, and a boxplot was used to identify and remove outliers (unrealistic or anomalous values) from the dataframe.
- *Average height*: 2.65 meters.
- *Range*: The data was filtered to include only values between 2.45 meters (min) and 3.6 meters (max).

- **`last_price` variable**

In [ ]:
len(data['last_price'].unique())

In [ ]:
data.boxplot(column='last_price');

In [ ]:
data.boxplot(column='last_price', showfliers=False);

In [ ]:
data['last_price'].sort_values()

In [ ]:
data['last_price'].describe()

In [ ]:
(data['last_price']/1000000).hist(range=(0, 20), bins=15);

In [ ]:
data = data.query('12190.00 < last_price < 20000000.00').reset_index(drop=True)

In [ ]:
data['last_price'].sort_values()

In [ ]:
data['last_price'].describe()

In [ ]:
data.boxplot(column='last_price');

In [ ]:
len(data['last_price'].unique())

In [ ]:
data.info()

**Conclusion:** The `last_price` variable was analyzed, and outliers (anomalously low and high property prices) were identified using a boxplot and removed from the main dataframe. Based on the boxplot and additional analysis, the average apartment price in the processed dataset is 4.6 million. The data was filtered to include a range between a minimum of 430,000 and a maximum of 20 million.

- **`city_centers_nearest` variable**

In [ ]:
len(data['city_centers_nearest'].unique())

In [ ]:
data.boxplot(column='city_centers_nearest');

In [ ]:
data.boxplot(column='city_centers_nearest', showfliers=False);

In [ ]:
data['city_centers_nearest'].describe()

In [ ]:
data[(data['city_centers_nearest'] > 2878.0) & 
    (data['city_centers_nearest'] < 40000.0)]['city_centers_nearest'].hist(bins=30);

In [ ]:
data = data.query('287.0 <= city_centers_nearest <= 36000.0 or city_centers_nearest.isna()').reset_index(drop=True)

In [ ]:
data.boxplot(column='city_centers_nearest');

In [ ]:
len(data['city_centers_nearest'].unique())

In [ ]:
data.info()

**Conclusion:** The `city_centers_nearest` variable was analyzed, and outliers (apartments anomalously distant from the center of Saint Petersburg) were identified using a boxplot and removed from the dataframe. Based on the boxplot and additional analysis, the average distance from the city center in the processed data is 13.321 km, with a minimum of 287 m and a maximum of 36 km.

- **`total_area` variable**

In [ ]:
len(data['total_area'].unique())

In [ ]:
data.boxplot(column='total_area');

In [ ]:
data.boxplot(column='total_area', showfliers=False);

In [ ]:
data['total_area'].sort_values()

In [ ]:
data['total_area'].describe()

In [ ]:
data[(data['total_area'] > 30.0) & (data['total_area'] < 160)]['total_area'].hist(bins=30);

In [ ]:
data = data.query('30.0 <= total_area <= 120.0').reset_index(drop=True)

In [ ]:
data['total_area'].sort_values()

In [ ]:
data.boxplot(column='total_area');

In [ ]:
len(data['total_area'].unique())

In [ ]:
data['total_area'].describe()

In [ ]:
data.info()

**Conclusion:** The `total_area` variable was analyzed, and outliers (apartments with anomalously small or large areas) were identified using a boxplot and removed from the dataframe. Based on the boxplot and additional analysis, the average apartment area in the processed data is 51 m², with a minimum of 30 m² and a maximum of 120 m².

## 3. Feature Engineering

In [ ]:
# Price per square meter (rounded to 2 decimal places)
data['price_per_sqm'] = round(data['last_price'] / data['total_area'], 2)

In [ ]:
# Day of the week of publication (0 = Monday, 1 = Tuesday, etc.)
data['weekday_exposition'] = data['first_day_exposition'].dt.weekday

In [ ]:
# Publication month
data['month_exposition'] = data['first_day_exposition'].dt.month

In [ ]:
# Publication year
data['year_exposition'] = data['first_day_exposition'].dt.year

In [ ]:
# Floor category (values: 'first', 'last', 'other')
def floor_group(row):
    if row['floor'] == 1:
        return 'first'
    if row['floor'] == row['floors_total']:
        return 'last'
    else:
        return 'other'
data['floor_group'] = data.apply(floor_group, axis=1)

In [ ]:
data['floor_group'].value_counts()

In [ ]:
# Distance to the city center in km (converted from meters and rounded to the nearest integer)
data['city_centers_nearest_km'] = round(data['city_centers_nearest']/1000)

In [ ]:
data.head(30)

**Conclusion:** Data preparation for hypothesis testing was completed in this section. The number of columns has increased by 10 since the start of the analysis (now totaling 32); these additional features will be utilized in the subsequent exploratory data analysis.

## 4. Exploratory Data Analysis (EDA)

### 4.1 Analysis and Description of Key Variables using Histograms

In [ ]:
# total_area
data.plot(y = 'total_area', kind = 'hist', bins = 30, grid=True, figsize = (7,5), range = (0,160))
plt.title('Total Apartment Area')
plt.xlabel('Square meters')
plt.ylabel('Number of apartments')
plt.show()

In [ ]:
data['total_area'].describe()

In [ ]:
# living_area
data.plot(y = 'living_area', kind = 'hist', bins = 70, grid=True, figsize = (7,5), range = (0,100))
plt.title('Living Area')
plt.xlabel('Living area in square meters')
plt.ylabel('Number of apartments')
plt.show()

In [ ]:
data['living_area'].describe()

In [ ]:
# kitchen_area
data.plot(y = 'kitchen_area', kind = 'hist', bins = 30, grid=True, figsize = (7,5), range = (0,50))
plt.title('Kitchen Area')
plt.xlabel('Kitchen area in square meters')
plt.ylabel('Number of apartments')
plt.show()

In [ ]:
data['kitchen_area'].describe()

In [ ]:
# last_price
data.plot(y = 'last_price', kind = 'hist', bins = 30, grid=True, figsize = (7,5), range = (0,20000000))
plt.title('Apartment Prices')
plt.xlabel('Price')
plt.ylabel('Number of apartments')
plt.show()

In [ ]:
data['last_price'].describe()

In [ ]:
# rooms
data.plot(y = 'rooms', kind = 'hist', bins = 7, grid=True, figsize = (7,5), range = (1,7))
plt.title('Rooms in apartments')
plt.xlabel('Number of rooms')
plt.ylabel('Number of apartments')
plt.show()

In [ ]:
data['rooms'].describe()

In [ ]:
# ceiling_height
data.plot(y = 'ceiling_height', kind = 'hist', bins = 20, grid=True, figsize = (7,5), range = (2,4))
plt.title('Ceiling Height')
plt.xlabel('Meters')
plt.ylabel('Number of apartments')
plt.show()

In [ ]:
data['ceiling_height'].describe()

In [ ]:
# floor_group
data.groupby('floor_group')['total_area'].count().plot( kind = 'bar', grid=True, figsize = (7,5))
plt.title('Floor Group')
plt.xlabel('Floor type')
plt.ylabel('Number of apartments')
plt.show()

In [ ]:
data['floor_group'].describe()

In [ ]:
# floors_total
data.plot(y = 'floors_total', kind = 'hist', bins = 20, grid=True, figsize = (7,5), range = (0,40))
plt.title('Total Floors in Building')
plt.xlabel('Floors')
plt.ylabel('Number of apartments')
plt.show()

In [ ]:
data['floors_total'].describe()

In [ ]:
# city_centers_nearest
data.plot(y = 'city_centers_nearest', kind = 'hist', bins = 50, grid=True, figsize = (7,5), range = (0,36000))
plt.title('Distance to City Center (m)')
plt.xlabel('Meters to city center')
plt.ylabel('Number of apartments')
plt.show()

In [ ]:
data['city_centers_nearest'].describe()

In [ ]:
# parks_nearest
data.plot(y = 'parks_nearest', kind = 'hist', bins = 40, grid=True, figsize = (7,5), range = (0,1500))
plt.title('Distance to Nearest Park')
plt.xlabel('Meters to nearest park')
plt.ylabel('Number of apartments')
plt.show()

In [ ]:
data['parks_nearest'].describe()

!!!


***Conclusions on Parameter Analysis***

- **Total Area**: The majority of apartments are under 100 m²; areas larger than this are rare. Although the original dataframe contained several anomalous outliers exceeding 200 m², the upper limit was restricted to 120 m² in the previous data cleaning step.
- **Living Area**: The most common living areas range from 15 to 50 m². Values above 60–65 m² are sporadic. The histogram displays two distinct peaks, likely corresponding to standard living areas for 1-room and 2-room apartments, respectively (followed by a smaller peak for 3-room apartments).
- **Kitchen Area**: Kitchen areas predominantly fall between 5 and 13 m². Areas exceeding 20 m² are rare, and those above 30 m² are isolated cases.
- **Apartment Price**: Prices mostly cluster around 3–5 million. Properties exceeding 15 million are scarce and appear anomalous, with some being significantly more expensive (noting that a portion of the extremely high-priced outliers was previously removed from the dataset).
- **Rooms**: The dataset primarily consists of 1-, 2-, and 3-room apartments, with a small proportion of 4-room properties. Listings with 5 or more rooms appear anomalous (even considering potential apartment combinations) and may represent separate houses.
- **Ceiling Height**: The majority of properties have ceiling heights between 2.5 and 3.1 meters. While values exceeding 3.5 meters appear anomalous, they may correspond to historic luxury buildings in Saint Petersburg, where ceilings can reach up to 5 meters.
- **Floor Group**: The "other" floor category is the most frequent, followed by "last" and "first." This is expected, as the majority of apartments are situated between the ground and top floors.
- **Total Floors in Building**: Five-story buildings represent the largest segment in the dataset. Distinct peaks are also observed at 9, 17, and 25 floors. This aligns with the architectural reality of Saint Petersburg, where five-story buildings are predominant.
- **Distance to City Center**: The highest volume of sales listings comes from residential areas on the outskirts (12,000–15,000 m), with a significant secondary peak observed at a distance of 5,000 meters from the city center.
- **Distance to Nearest Park**: The number of listings decreases as the distance to the nearest park increases. The average distance to a park is approximately 200–700 meters.

**Summary**: The typical listing describes a 1- or 2-room apartment in a 5-story building, with an area of 40 m², a 7 m² kitchen, 2.5-meter ceilings, located approximately 13 km from the city center.

### 4.2 Analysis of Sale Duration

In [ ]:
data['days_exposition'].describe()

In [ ]:
# plotting the histogram
data.plot(y = 'days_exposition', kind = 'hist', bins = 100, grid=True, figsize = (7,5), range = (0, 500))
plt.title('Time to Sell Apartments (days)')
plt.xlabel('Days (Time to sell)')
plt.ylabel('Number of apartments')
plt.show()

In [ ]:
# calculating the mean and median
print('Time to sell (mean):', round(data['days_exposition'].mean(), 3))
print('Time to sell (median):', data['days_exposition'].median())

**Conclusion:** The median time to sell an apartment is 93 days (approximately 3 months), while the average (mean) time is 176 days (approximately 6 months). Typically, a standard property sale takes 2–3 months, depending on associated factors (area, price, and distance from the city center).

- **Fast Sales:** Sales completed within 45 days (the first quartile) can be considered "fast." The significant peak observed in this range suggests that listings might be automatically removed or unpublished after this period, creating the appearance of a rapid sale in the dataset.

- **Unusually Long Sales:** Transactions taking 224 days or more (beyond the third quartile) are considered unusually long. These cases may represent properties that remained unsold for extended periods; for example, the maximum duration recorded in the dataset is 1,580 days (~4.3 years).

### 4.3 Analysis of factors influencing the total apartment price

In [ ]:
# total area vs. last price
data.plot(x = 'total_area', y = 'last_price', kind = 'scatter', grid=True, figsize=(10,5), alpha=0.3)
plt.title('Correlation between Total Area and Apartment Price')
plt.ticklabel_format(style='plain')
plt.xlabel('Total area')
plt.ylabel('Price')
plt.show()

In [ ]:
# living area vs. last price
data.plot(x = 'living_area', y = 'last_price', kind = 'scatter', grid=True, figsize=(10,5), alpha=0.3)
plt.title('Correlation between Living Area and Apartment Price')
plt.ticklabel_format(style='plain')
plt.xlabel('Living area')
plt.ylabel('Price')
plt.show()

In [ ]:
data['kitchen_area'].describe()

In [ ]:
data.boxplot(column='kitchen_area')

In [ ]:
# kitchen area vs. last price
data.loc[data['kitchen_area'] < 30].plot(x = 'kitchen_area', y = 'last_price', kind = 'scatter', 
                                         grid=True, figsize=(10, 5), alpha=0.3)
plt.title('Correlation between Kitchen Area and Apartment Price')
plt.ticklabel_format(style='plain')
plt.xlabel('Kitchen area')
plt.ylabel('Price')
plt.show()

In [ ]:
# rooms vs. last price
data.groupby('rooms')['last_price'].median().plot(kind = 'bar', grid=True, figsize = (7,5))
plt.title('Correlation between Number of Rooms and Apartment Price')
plt.xlabel('Rooms')
plt.ylabel('Price')
plt.show()

In [ ]:
# floor group vs. price
data.groupby('floor_group')['last_price'].median().plot(kind = 'bar', grid=True, figsize = (7,5))
plt.title('Median Apartment Price by Floor Group')
plt.xlabel('Floor group')
plt.ylabel('Price')
plt.show()

In [ ]:
# weekday exposition vs. price
(
    data.pivot_table(index='weekday_exposition', values='last_price')
    .plot(grid=True, figsize=(7,5), style = '-o')
)
plt.title('Correlation between Publication Day and Apartment Price')
plt.xlabel('Publication Day')
plt.ylabel('Price')
plt.show()

In [ ]:
# month exposition vs. price
(
    data.pivot_table(index='month_exposition', values='last_price')
    .plot(grid=True, figsize=(7,5), style = '-o')
)
plt.title('Correlation between Publication Month and Apartment Price')
plt.xlabel('Publication Month')
plt.ylabel('Price')
plt.show()

In [ ]:
# year exposition vs. price
(
    data.pivot_table(index='year_exposition', values='last_price')
    .plot(grid=True, figsize=(7,5), style = '-o')
)
plt.title('Correlation between Publication Year and Apartment Price')
plt.xlabel('Publication Year')
plt.ylabel('Price')
plt.show()

In [ ]:
fig = plt.figure(figsize=(10, 1))
sns.heatmap(data[['last_price',
                    'total_area',
                      'price_per_sqm',
                      'living_area',
                      'kitchen_area',
                      'rooms',
                      'ceiling_height',
                      'floor',
                      'floors_total',
                      'balcony', 
                      'city_centers_nearest',
                      'city_centers_nearest_km'
                 ]].corr().loc[['last_price']].drop(columns='last_price'), annot=True);

**Conclusions on the Analysis of Factors Influencing Total Apartment Price**

- **Total Area**: A strong positive correlation exists between total area and price; as the area increases, the price rises. The correlation coefficient is high (0.73).
- **Living Area**: The living area also influences the apartment price, showing a moderate correlation of 0.60.
- Kitchen Area: The correlation between kitchen area and price (0.51) is lower than that of the total and living areas. A potential reason is that kitchen size may be a less critical factor for buyers compared to overall and living space.
- **Number of Rooms**: While the number of rooms positively influences price, the correlation (0.43) is weaker than that of the area variables. This suggests the existence of apartments with fewer but larger rooms, or that centrally located apartments may have fewer rooms yet command higher prices due to location.
- **Floor Type**: On average, prices for ground-floor apartments are 20% lower than those on other floors (excluding the top floor). Apartments on the top floor are slightly more expensive than those on the ground floor, while properties located between the ground and top floors command the highest prices.
- **Publication Date Analysis**:

    - *Day of the Week:* Listings published on Tuesdays command the highest prices, while those published on Saturdays and Sundays are the cheapest.
    - *Month:* The highest prices are observed in April and September, while June exhibits the lowest prices.
    - *Year:* Prices peaked in 2014 (likely influenced by the economic aftermath of the 2008–2013 period), followed by a significant decline between 2016 and 2018. In 2019, prices began to rise again as the economy stabilized.

### 4.4 Analysis of the Top 10 Localities by Number of Listings and Price per Square Meter

In [ ]:
# create a pivot table showing the number of listings and average price per square meter for these localities
top10_locality_names = data.pivot_table(index = 'locality_clean_name', values = 'price_per_sqm',
                                       aggfunc=['count', 'mean'])
top10_locality_names.columns = ['count', 'mean']
top10_locality_names = tip10_locality_names.sort_values('count', ascending = False).head(10)
top10_locality_names

In [ ]:
top10_locality_names.groupby('locality_clean_name')['mean'].mean().plot(kind = 'bar', grid=True, figsize = (7,5))
plt.title('Top 10 Localities with the Most Listings')
plt.xlabel('Locality Name')
plt.ylabel('Average Price per m²')
plt.show()

In [ ]:
# highlighting localities with the highest and lowest price per square meter
# highest price
top10_locality_names[top10_locality_names['mean']==top10_locality_names['mean'].max()]

In [ ]:
# lowest price
top10_locality_names[top10_locality_names['mean']==top10_locality_names['mean'].min()]

**Conclusion:** As expected, the highest average price per square meter is found in Saint Petersburg. This is followed by Pushkin, which features numerous parks and modern developments. Among the top 10, the lowest prices are observed in Gatchina and Vsevolozhsk (likely due to the commute distance to Saint Petersburg), as well as Vyborg (10th on the list), which is situated at a significant distance from the city.

### 4.5 Analysis of Price per Square Meter in Saint Petersburg per Kilometer of Distance from the Center

In [ ]:
# average prices calculated for every kilometer of distance (1 km, 2 km, etc.) from the center
last_price_per_km = (data.query('locality_clean_name == "Санкт-Петербург"')
                    .pivot_table(index='city_centers_nearest_km', values='last_price', aggfunc='mean'))
last_price_per_km

In [ ]:
m2_price_per_km = (data.query('locality_clean_name == "Санкт-Петербург"')
                    .pivot_table(index='city_centers_nearest_km', values='price_per_sqm', aggfunc='mean'))
m2_price_per_km

In [ ]:
data[data['locality_clean_name'] == 'Санкт-Петербург']['city_center_nearest_km'].describe()

In [ ]:
data.query('locality_clean_name == "Санкт-Петербург"').boxplot(column='city_center_nearest_km');

In [ ]:
data.query('locality_clean_name == "Санкт-Петербург" and city_center_nearest_km == 27')

In [ ]:
# plotting the change in average price for each kilometer from the center of St. Petersburg
(
    data.query('locality_clean_name == "Санкт-Петербург"')
    .pivot_table(index='city_centers_nearest_km', values='last_price', aggfunc='mean')
    .plot(grid=True, figsize=(12,5))
)
plt.title('Average Total Apartment Price vs. Distance to Center (St. Petersburg)')
plt.ticklabel_format(style='plain')
plt.xlabel('Distance from city center (km)')
plt.ylabel('Average apartment price')
plt.show()

In [ ]:
(
    data.query('locality_clean_name == "Санкт-Петербург"')
    .pivot_table(index='city_centers_nearest_km', values='price_per_sqm', aggfunc='mean')
    .plot(grid=True, figsize=(12,5))
)
plt.title('Average Price per m² vs. Distance to Center (St. Petersburg)')
plt.ticklabel_format(style='plain')
plt.xlabel('Distance from city center (km)')
plt.ylabel('Average price per m²')
plt.show()

**Conclusion:** The most distant property is located 29 km from the center of Saint Petersburg.

Particular attention was drawn to the 27 km mark, where housing prices appeared nearly double those at 26 km and 28 km (suggesting either luxury real estate or insufficient data). An examination of the data rows for the 27 km distance revealed that the observed peak is due to a small sample size (only 2 properties), which resulted in an inflated average price.

Based on the first pivot table and graph, apartment prices are highest within the first 10 km from the center and gradually decrease thereafter. The second pivot table (km/price per m²) and graph indicate noticeable peaks in the average price per square meter at 1–7 km from the city center, followed by a gradual decline.

## 5. Conclusion

Real estate listings for apartments in Saint Petersburg and its surrounding areas were analyzed in this study. During the data preparation phase, data preprocessing was performed (identifying missing values, correcting data types, and eliminating implicit duplicates and anomalies), and new columns containing necessary parameters were engineered to facilitate exploratory analysis.

The exploratory data analysis yielded the following conclusions:

- **Time to Sell (Days on Market)**

The median sale time is 93 days. Sales completed in under 45 days are considered fast (likely due to automatic delisting), while those taking longer than 224 days are considered slow. A significant number of apartments were sold within just a few days of publication, whereas some listings remained on the market for several dozen months (reaching up to ~4 years).

- **Factors Influencing Price**

The total area has the most significant impact on the total apartment price, followed by the price per square meter, living area, kitchen area, number of rooms, and ceiling height.

- *Floor Level:* Ground-floor apartments are cheaper than those on other floors. Top-floor apartments are more expensive than those on the ground floor but cheaper than those on middle floors.

- *Location:* Proximity to the city center correlates with higher prices.

- *Timing:* Listings published on weekdays are priced higher than those published on weekends. Prices are highest in April and September and lowest in June.

- *Yearly Trend:* Apartments sold in 2014 commanded the highest prices (likely due to post-crisis economic conditions); subsequently, prices declined before beginning to recover in 2019.

- **Price per Square Meter in Top 10 Localities**

The highest price per square meter is found in Saint Petersburg (108,326), while the lowest is in Vyborg (58,027).

- **Factors Influencing Price in Central Saint Petersburg**

The most expensive apartments are located in the city center. Prices decrease as the distance from the center increases up to 7 kilometers; beyond this radius, price shows no significant dependence on distance.